## Step 1: Setup – Load the Cleaned Corpus

### 1.1 Install required libraries

In [1]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


### 1.2 Import libraries and shared settings

In [3]:
import os
import time

import numpy as np
import pandas as pd

from representations import (
    load_cleaned_corpus, load_tokenized_reviews,
    COUNT_SETTINGS, EMBEDDING_SETTINGS, MODELS_DIR,
)

### 1.3 Load the cleaned dataset from Assignment 1

In [4]:
corpus_df = load_cleaned_corpus()

print("Shape (rows, columns):", corpus_df.shape)
print("Columns:", corpus_df.columns.tolist())
corpus_df.head()

Shape (rows, columns): (50000, 3)
Columns: ['review_id', 'sentiment', 'cleaned_text']


,review_id,sentiment,cleaned_text
0,0,positive,one reviewer mention watch 1 oz episode 'll ho...
1,1,positive,wonderful little production . film technique u...
2,2,positive,think wonderful way spend time hot summer week...
3,3,negative,basically 's family little boy ( jake ) think ...
4,4,positive,petter mattei 's `` love time money '' visuall...


### 1.4 Corpus summary

In [5]:
tokenized_reviews = load_tokenized_reviews(corpus_df)

total_tokens = sum(len(tokens) for tokens in tokenized_reviews)
corpus_vocabulary = set()
for tokens in tokenized_reviews:
    corpus_vocabulary.update(tokens)

print("Number of documents:       ", len(corpus_df))
print("Total tokens:              ", total_tokens)
print("Vocabulary size:           ", len(corpus_vocabulary))
print("Average tokens per review: ", round(total_tokens / len(corpus_df), 1))
print("\nSentiment distribution:")
print(corpus_df["sentiment"].value_counts())

Number of documents:        50000
Total tokens:               7765868
Vocabulary size:            142754
Average tokens per review:  155.3

Sentiment distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


## Step 2: Bag of Words (A1)

### 2.1 Build the document-term count matrix

In [6]:
from sklearn.feature_extraction.text import CountVectorizer

bow_vectorizer = CountVectorizer(**COUNT_SETTINGS)
bow_matrix = bow_vectorizer.fit_transform(corpus_df["cleaned_text"])

bow_feature_names = bow_vectorizer.get_feature_names_out()
print("Matrix type:", type(bow_matrix).__name__)
print("Number of features (vocabulary):", len(bow_feature_names))

Matrix type: csr_matrix
Number of features (vocabulary): 142754


### 2.2 Shape and sparsity

In [7]:
n_documents, n_features = bow_matrix.shape
total_cells = n_documents * n_features
non_zero_cells = bow_matrix.nnz
zero_cells = total_cells - non_zero_cells
bow_sparsity = zero_cells / total_cells * 100

print("Shape (documents, features):", bow_matrix.shape)
print("Total cells:                ", f"{total_cells:,}")
print("Non-zero cells:             ", f"{non_zero_cells:,}")
print("Zero cells:                 ", f"{zero_cells:,}")
print(f"Sparsity (% of zero cells):  {bow_sparsity:.4f}%")
print(f"Average non-zero features per document: {non_zero_cells / n_documents:.1f}")

Shape (documents, features): (50000, 142754)
Total cells:                 7,137,700,000
Non-zero cells:              5,133,224
Zero cells:                  7,132,566,776
Sparsity (% of zero cells):  99.9281%
Average non-zero features per document: 102.7


### 2.3 Memory: sparse vs dense matrix

In [8]:
sparse_mb = (bow_matrix.data.nbytes + bow_matrix.indices.nbytes + bow_matrix.indptr.nbytes) / 1024**2
dense_gb = total_cells * bow_matrix.dtype.itemsize / 1024**3

print(f"Sparse matrix memory:            {sparse_mb:,.1f} MB")
print(f"Same matrix stored dense would need: {dense_gb:,.1f} GB")

Sparse matrix memory:            58.9 MB
Same matrix stored dense would need: 53.2 GB


### 2.4 Check: the counts add up to the number of tokens

In [9]:
row_totals = np.asarray(bow_matrix.sum(axis=1)).ravel()
token_counts = corpus_df["cleaned_text"].str.split().str.len().to_numpy()

print("Total count in matrix:", row_totals.sum())
print("Total tokens in corpus:", token_counts.sum())
print("Every row total equals its number of tokens:", bool((row_totals == token_counts).all()))

Total count in matrix: 7765868
Total tokens in corpus: 7765868
Every row total equals its number of tokens: True


### 2.5 One review as a Bag of Words vector

In [10]:
example_index = 0
example_row = bow_matrix[example_index]

example_counts = pd.DataFrame({
    "word": bow_feature_names[example_row.indices],
    "count": example_row.data,
}).sort_values("count", ascending=False)

print("Review", example_index, "(first 200 characters):")
print(corpus_df.loc[example_index, "cleaned_text"][:200])
print("\nNon-zero features in this review:", example_row.nnz, "out of", n_features)
print("\nMost frequent words in this review:")
print(example_counts.head(15).to_string(index=False))

Review 0 (first 200 characters):
one reviewer mention watch 1 oz episode 'll hook . right , exactly happen . first thing strike oz brutality unflinching scene violence , set right word go . trust , not show faint hearted timid . show

Non-zero features in this review: 146 out of 142754

Most frequent words in this review:
    word  count
       ,     26
       .     13
      oz      6
    show      4
violence      4
     ...      3
  prison      3
     not      3
  forget      3
     n't      3
     get      3
     'll      3
   right      2
 episode      2
    city      2


### 2.6 Most frequent words in the whole corpus

In [11]:
word_totals = np.asarray(bow_matrix.sum(axis=0)).ravel()
document_frequency = np.asarray((bow_matrix > 0).sum(axis=0)).ravel()

top_words = pd.DataFrame({
    "word": bow_feature_names,
    "total count": word_totals,
    "documents containing it": document_frequency,
}).sort_values("total count", ascending=False)

print(top_words.head(20).to_string(index=False))

 word  total count  documents containing it
    ,       543849                    48051
    .       535267                    49513
   's       122984                    35673
movie       101535                    32151
 film        93513                    29455
    )        71243                    26693
    (        69579                    26452
  n't        66709                    29736
   ``        66304                    23221
   ''        66034                    23190
  not        62414                    30144
  one        53990                    28488
 make        43544                    25120
 like        42427                    23978
  see        40933                    24353
    !        38284                    17349
  get        35135                    21005
 good        33946                    21072
 time        30545                    19906
    ?        29848                    14791


## Step 3: TF-IDF (A2)

### 3.1 Build the TF-IDF matrix

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(**COUNT_SETTINGS)
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus_df["cleaned_text"])
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()

print("Settings used:")
print("  use_idf:     ", tfidf_vectorizer.use_idf)
print("  smooth_idf:  ", tfidf_vectorizer.smooth_idf)
print("  sublinear_tf:", tfidf_vectorizer.sublinear_tf)
print("  norm:        ", tfidf_vectorizer.norm)

Settings used:
  use_idf:      True
  smooth_idf:   True
  sublinear_tf: False
  norm:         l2


### 3.2 Shape and sparsity (same corpus as Bag of Words)

In [13]:
tfidf_sparsity = (1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])) * 100

print("Shape (documents, features):", tfidf_matrix.shape)
print("Non-zero cells:             ", f"{tfidf_matrix.nnz:,}")
print(f"Sparsity (% of zero cells):  {tfidf_sparsity:.4f}%")
print("\nSame vocabulary as Bag of Words:", bool((tfidf_feature_names == bow_feature_names).all()))
print("Same non-zero cells as Bag of Words:", tfidf_matrix.nnz == bow_matrix.nnz)

Shape (documents, features): (50000, 142754)
Non-zero cells:              5,133,224
Sparsity (% of zero cells):  99.9281%

Same vocabulary as Bag of Words: True
Same non-zero cells as Bag of Words: True


### 3.3 IDF values: most common vs rarest words

In [14]:
idf_table = pd.DataFrame({
    "word": tfidf_feature_names,
    "documents containing it": document_frequency,
    "idf": tfidf_vectorizer.idf_,
}).sort_values("idf")

print("LOWEST IDF (words found in most reviews):")
print(idf_table.head(10).to_string(index=False))
print("\nHIGHEST IDF (words found in only one review):")
print(idf_table.tail(5).to_string(index=False))
print("\nWords that appear in only one review:", (document_frequency == 1).sum())

LOWEST IDF (words found in most reviews):
 word  documents containing it      idf
    .                    49513 1.009788
    ,                    48051 1.039759
   's                    35673 1.337621
movie                    32151 1.441568
  not                    30144 1.506024
  n't                    29736 1.519651
 film                    29455 1.529145
  one                    28488 1.562525
    )                    26693 1.627604
    (                    26452 1.636673

HIGHEST IDF (words found in only one review):
                              word  documents containing it       idf
                    halloween-mask                        1 11.126651
halloween-costume-for-a-6-year-old                        1 11.126651
                   halloween-based                        1 11.126651
                  halloween/friday                        1 11.126651
                             כרמון                        1 11.126651

Words that appear in only one review: 81805


### 3.4 Check the IDF formula for one word

In [15]:
n_docs = tfidf_matrix.shape[0]
word = "."
df_word = document_frequency[tfidf_vectorizer.vocabulary_[word]]
idf_by_formula = np.log((1 + n_docs) / (1 + df_word)) + 1

print(f"Word: '{word}' appears in {df_word} of {n_docs} reviews")
print(f"idf = ln((1 + {n_docs}) / (1 + {df_word})) + 1 = {idf_by_formula:.6f}")
print(f"scikit-learn idf_ value                     = {tfidf_vectorizer.idf_[tfidf_vectorizer.vocabulary_[word]]:.6f}")

Word: '.' appears in 49513 of 50000 reviews
idf = ln((1 + 50000) / (1 + 49513)) + 1 = 1.009788
scikit-learn idf_ value                     = 1.009788


### 3.5 Concrete example: high and low TF-IDF weights in one review

In [16]:
example_index = 0
tfidf_row = tfidf_matrix[example_index]

example_weights = pd.DataFrame({
    "word": tfidf_feature_names[tfidf_row.indices],
    "count in review": bow_matrix[example_index, tfidf_row.indices].toarray().ravel(),
    "documents containing it": document_frequency[tfidf_row.indices],
    "idf": tfidf_vectorizer.idf_[tfidf_row.indices].round(3),
    "tf-idf weight": tfidf_row.data.round(4),
}).sort_values("tf-idf weight", ascending=False)

print("Review", example_index, "(first 200 characters):")
print(corpus_df.loc[example_index, "cleaned_text"][:200])
print("\nHIGH weight words:")
print(example_weights.head(10).to_string(index=False))
print("\nLOW weight words (driven toward zero):")
print(example_weights.tail(10).to_string(index=False))

Review 0 (first 200 characters):
one reviewer mention watch 1 oz episode 'll hook . right , exactly happen . first thing strike oz brutality unflinching scene violence , set right word go . trust , not show faint hearted timid . show

HIGH weight words:
       word  count in review  documents containing it    idf  tf-idf weight
         oz                6                      168  6.690         0.4490
          ,               26                    48051  1.040         0.3024
   violence                4                     1659  4.405         0.1971
     prison                3                      634  5.366         0.1801
          .               13                    49513  1.010         0.1468
     forget                3                     2020  4.208         0.1412
penitentary                1                        2 10.721         0.1199
    inwards                1                        2 10.721         0.1199
        'll                3                     4757  3.352  

### 3.6 Same words: Bag of Words count vs TF-IDF weight

In [17]:
compare_words = [",", ".", "not", "show", "oz", "prison", "violence"]
compare_words = [w for w in compare_words if w in set(example_weights["word"])]

comparison = example_weights.set_index("word").loc[compare_words, ["count in review", "documents containing it", "tf-idf weight"]]
print(comparison.to_string())

          count in review  documents containing it  tf-idf weight
word                                                             
,                      26                    48051         0.3024
.                      13                    49513         0.1468
not                     3                    30144         0.0505
show                    4                    11756         0.1095
oz                      6                      168         0.4490
prison                  3                      634         0.1801
violence                4                     1659         0.1971


### 3.7 Check: every review vector has length 1 (L2 normalization)

In [18]:
row_lengths = np.sqrt(np.asarray(tfidf_matrix.multiply(tfidf_matrix).sum(axis=1)).ravel())

print("Smallest row length:", round(row_lengths.min(), 6))
print("Largest row length: ", round(row_lengths.max(), 6))

Smallest row length: 1.0
Largest row length:  1.0


## Step 4: N-grams (A3)

### 4.1 Sample used for N-grams

In [19]:
# Bigrams and trigrams create millions of features, so we use a fixed random sample of 10,000 reviews
ngram_sample = corpus_df["cleaned_text"].sample(n=10000, random_state=42)

sample_tokens = ngram_sample.str.split().str.len().sum()
print("Reviews in sample:", len(ngram_sample))
print("Tokens in sample: ", sample_tokens)

Reviews in sample: 10000
Tokens in sample:  1559125


### 4.2 Vocabulary growth with n (unigrams, bigrams, trigrams)

In [20]:
ngram_settings = [
    ("Unigrams", (1, 1)),
    ("Bigrams", (2, 2)),
    ("Trigrams", (3, 3)),
    ("Uni + Bi", (1, 2)),
    ("Uni + Bi + Tri", (1, 3)),
]

ngram_results = []
ngram_matrices = {}
ngram_vectorizers = {}

for name, ngram_range in ngram_settings:
    start_time = time.time()
    vectorizer = CountVectorizer(**COUNT_SETTINGS, ngram_range=ngram_range)
    matrix = vectorizer.fit_transform(ngram_sample)
    n_docs, n_feat = matrix.shape
    memory_mb = (matrix.data.nbytes + matrix.indices.nbytes + matrix.indptr.nbytes) / 1024**2

    ngram_results.append({
        "Setting": name,
        "ngram_range": str(ngram_range),
        "Features": n_feat,
        "Non-zero cells": matrix.nnz,
        "Avg non-zero per review": round(matrix.nnz / n_docs, 1),
        "Sparsity %": round((1 - matrix.nnz / (n_docs * n_feat)) * 100, 4),
        "Sparse memory (MB)": round(memory_mb, 1),
        "Time (s)": round(time.time() - start_time, 1),
    })
    ngram_matrices[name] = matrix
    ngram_vectorizers[name] = vectorizer

ngram_table = pd.DataFrame(ngram_results)
ngram_table["Features vs unigrams"] = (ngram_table["Features"] / ngram_table.loc[0, "Features"]).round(1).astype(str) + "x"
print(ngram_table.to_string(index=False))

       Setting ngram_range  Features  Non-zero cells  Avg non-zero per review  Sparsity %  Sparse memory (MB)  Time (s) Features vs unigrams
      Unigrams      (1, 1)     58873         1030846                    103.1     99.8249                11.8       2.6                 1.0x
       Bigrams      (2, 2)    692749         1489183                    148.9     99.9785                17.1       9.9                11.8x
      Trigrams      (3, 3)   1327447         1528573                    152.9     99.9885                17.5      15.9                22.5x
      Uni + Bi      (1, 2)    751622         2520029                    252.0     99.9665                28.9      13.0                12.8x
Uni + Bi + Tri      (1, 3)   2079069         4048602                    404.9     99.9805                46.4      29.7                35.3x


### 4.3 Long tail: n-grams that appear in only one review

In [21]:
for name in ["Unigrams", "Bigrams", "Trigrams"]:
    matrix = ngram_matrices[name]
    doc_freq = np.asarray((matrix > 0).sum(axis=0)).ravel()
    only_once = (doc_freq == 1).sum()
    print(f"{name:9} features: {matrix.shape[1]:>9,}   in only 1 review: {only_once:>9,} ({only_once / matrix.shape[1] * 100:.1f}%)")

Unigrams  features:    58,873   in only 1 review:    31,622 (53.7%)
Bigrams   features:   692,749   in only 1 review:   554,961 (80.1%)
Trigrams  features: 1,327,447   in only 1 review: 1,245,724 (93.8%)


### 4.4 Sparsity trade-off: removing n-grams seen in only one review (min_df=2)

In [22]:
for min_df in [1, 2, 5]:
    vectorizer = CountVectorizer(**COUNT_SETTINGS, ngram_range=(1, 3), min_df=min_df)
    matrix = vectorizer.fit_transform(ngram_sample)
    sparsity = (1 - matrix.nnz / (matrix.shape[0] * matrix.shape[1])) * 100
    print(f"Uni + Bi + Tri, min_df={min_df}:  features = {matrix.shape[1]:>9,}   "
          f"avg non-zero per review = {matrix.nnz / matrix.shape[0]:6.1f}   sparsity = {sparsity:.4f}%")

Uni + Bi + Tri, min_df=1:  features = 2,079,069   avg non-zero per review =  404.9   sparsity = 99.9805%
Uni + Bi + Tri, min_df=2:  features =   246,762   avg non-zero per review =  221.6   sparsity = 99.9102%
Uni + Bi + Tri, min_df=5:  features =    61,973   avg non-zero per review =  176.4   sparsity = 99.7154%


### 4.5 Most frequent bigrams and trigrams

In [23]:
def is_word_ngram(ngram):
    # True if every token in the n-gram contains a letter (hides n-grams with "." or "," in this list only)
    return all(any(ch.isalpha() for ch in token) for token in ngram.split())


for name in ["Bigrams", "Trigrams"]:
    matrix = ngram_matrices[name]
    totals = np.asarray(matrix.sum(axis=0)).ravel()
    top = pd.DataFrame({
        "ngram": ngram_vectorizers[name].get_feature_names_out(),
        "count": totals,
    }).sort_values("count", ascending=False)
    top = top[top["ngram"].apply(is_word_ngram)]
    print(f"Top 15 {name.lower()} (word-only):")
    print(top.head(15).to_string(index=False))
    print()

Top 15 bigrams (word-only):
      ngram  count
     ca n't   1532
     's not    957
  look like    703
  could n't    611
   n't know    596
    've see    558
   n't even    554
   ever see    548
watch movie    533
    film 's    528
  see movie    510
     wo n't    510
  bad movie    473
 make movie    452
    n't get    415

Top 15 trigrams (word-only):
          ngram  count
   've ever see    213
 ca n't believe    107
 movie ever see     96
 bad movie ever     86
    ca n't help     86
    'm not sure     85
 n't waste time     85
 movie 've ever     75
movie ever make     61
  make no sense     60
    ca n't even     60
     ca n't get     58
 film ever make     57
     ca n't say     56
  new york city     53



### 4.6 Negation bigrams: meaning that single words miss

In [24]:
bigram_names = ngram_vectorizers["Bigrams"].get_feature_names_out()
bigram_totals = np.asarray(ngram_matrices["Bigrams"].sum(axis=0)).ravel()

negation_bigrams = pd.DataFrame({"bigram": bigram_names, "count": bigram_totals})
negation_bigrams = negation_bigrams[negation_bigrams["bigram"].str.match(r"^(not|n't|no|never) ")
                                    & negation_bigrams["bigram"].apply(is_word_ngram)]
print(negation_bigrams.sort_values("count", ascending=False).head(15).to_string(index=False))

    bigram  count
  n't know    596
  n't even    554
   n't get    415
   n't see    394
  not even    354
    no one    334
 n't think    319
n't really    311
  n't make    250
  not good    236
  n't like    227
  not much    227
  n't help    213
 never see    211
  n't want    210


## Step 5: Hand-worked Numerical (A4)

### 5.1 Toy sub-sample (4 short documents in the same cleaned format)

In [25]:
toy_docs = [
    "movie not good",
    "good movie good act",
    "bad movie not good",
    "great act",
]
toy_names = ["D1", "D2", "D3", "D4"]

for name, doc in zip(toy_names, toy_docs):
    print(f"{name}: {doc}")

D1: movie not good
D2: good movie good act
D3: bad movie not good
D4: great act


### 5.2 Library output: TF-IDF matrix of the toy documents

In [26]:
toy_tfidf_vectorizer = TfidfVectorizer(**COUNT_SETTINGS)
toy_tfidf = toy_tfidf_vectorizer.fit_transform(toy_docs)
toy_vocab = toy_tfidf_vectorizer.get_feature_names_out()

print("Vocabulary:", list(toy_vocab))
print("\nIDF from library:")
print(pd.Series(toy_tfidf_vectorizer.idf_, index=toy_vocab).round(6).to_string())
print("\nTF-IDF matrix from library:")
print(pd.DataFrame(toy_tfidf.toarray(), index=toy_names, columns=toy_vocab).round(6).to_string())

Vocabulary: ['act', 'bad', 'good', 'great', 'movie', 'not']

IDF from library:
act      1.510826
bad      1.916291
good     1.223144
great    1.916291
movie    1.223144
not      1.510826

TF-IDF matrix from library:
         act       bad      good     great     movie       not
D1  0.000000  0.000000  0.532570  0.000000  0.532570  0.657829
D2  0.483529  0.000000  0.782917  0.000000  0.391459  0.000000
D3  0.000000  0.640655  0.408922  0.000000  0.408922  0.505100
D4  0.619130  0.000000  0.000000  0.785288  0.000000  0.000000


### 5.3 Hand calculation: TF-IDF weight of "good" in D2

In [27]:
import math

N = len(toy_docs)
target_doc, target_word = 1, "good"          # D2, word "good"
doc_tokens = toy_docs[target_doc].split()
toy_tokens = [doc.split() for doc in toy_docs]

print(f"N = {N} documents,  target: '{target_word}' in {toy_names[target_doc]} = '{toy_docs[target_doc]}'")

print("\nStep 1 - term counts (tf) in D2:")
tf = {word: doc_tokens.count(word) for word in sorted(set(doc_tokens))}
for word, count in tf.items():
    print(f"  tf({word}) = {count}")

print("\nStep 2 - document frequency (df) and idf = ln((1 + N) / (1 + df)) + 1:")
idf = {}
for word in tf:
    df_word = sum(word in tokens for tokens in toy_tokens)
    idf[word] = math.log((1 + N) / (1 + df_word)) + 1
    print(f"  df({word}) = {df_word}   idf({word}) = ln({1 + N}/{1 + df_word}) + 1 = "
          f"{math.log((1 + N) / (1 + df_word)):.6f} + 1 = {idf[word]:.6f}")

print("\nStep 3 - raw tf-idf = tf x idf:")
raw = {}
for word in tf:
    raw[word] = tf[word] * idf[word]
    print(f"  {word}: {tf[word]} x {idf[word]:.6f} = {raw[word]:.6f}")

print("\nStep 4 - L2 norm of D2 = sqrt(sum of squares):")
squares = {word: value ** 2 for word, value in raw.items()}
for word, value in squares.items():
    print(f"  {word}: {raw[word]:.6f}^2 = {value:.6f}")
sum_squares = sum(squares.values())
norm = math.sqrt(sum_squares)
print(f"  sum = {' + '.join(f'{v:.6f}' for v in squares.values())} = {sum_squares:.6f}")
print(f"  norm = sqrt({sum_squares:.6f}) = {norm:.6f}")

hand_value = raw[target_word] / norm
library_value = toy_tfidf[target_doc, toy_tfidf_vectorizer.vocabulary_[target_word]]
print(f"\nStep 5 - final weight = {raw[target_word]:.6f} / {norm:.6f} = {hand_value:.6f}")
print(f"\nHand calculation:  {hand_value:.6f}")
print(f"Library (sklearn): {library_value:.6f}")
print("Match:", math.isclose(hand_value, library_value, rel_tol=1e-9))

N = 4 documents,  target: 'good' in D2 = 'good movie good act'

Step 1 - term counts (tf) in D2:
  tf(act) = 1
  tf(good) = 2
  tf(movie) = 1

Step 2 - document frequency (df) and idf = ln((1 + N) / (1 + df)) + 1:
  df(act) = 2   idf(act) = ln(5/3) + 1 = 0.510826 + 1 = 1.510826
  df(good) = 3   idf(good) = ln(5/4) + 1 = 0.223144 + 1 = 1.223144
  df(movie) = 3   idf(movie) = ln(5/4) + 1 = 0.223144 + 1 = 1.223144

Step 3 - raw tf-idf = tf x idf:
  act: 1 x 1.510826 = 1.510826
  good: 2 x 1.223144 = 2.446287
  movie: 1 x 1.223144 = 1.223144

Step 4 - L2 norm of D2 = sqrt(sum of squares):
  act: 1.510826^2 = 2.282594
  good: 2.446287^2 = 5.984321
  movie: 1.223144^2 = 1.496080
  sum = 2.282594 + 5.984321 + 1.496080 = 9.762995
  norm = sqrt(9.762995) = 3.124579

Step 5 - final weight = 2.446287 / 3.124579 = 0.782917

Hand calculation:  0.782917
Library (sklearn): 0.782917
Match: True


### 5.4 Hand calculation: the bigram "not good"

In [28]:
target_bigram = "not good"

print("Step 1 - list the bigrams (pairs of neighbouring tokens) in each document:")
hand_bigram_counts = {}
for name, tokens in zip(toy_names, toy_tokens):
    bigrams = [f"{a} {b}" for a, b in zip(tokens, tokens[1:])]
    hand_bigram_counts[name] = bigrams.count(target_bigram)
    print(f"  {name}: {bigrams}  ->  '{target_bigram}' appears {hand_bigram_counts[name]} time(s)")
hand_total = sum(hand_bigram_counts.values())
print(f"\nStep 2 - total count of '{target_bigram}' = "
      f"{' + '.join(str(c) for c in hand_bigram_counts.values())} = {hand_total}")

toy_bigram_vectorizer = CountVectorizer(**COUNT_SETTINGS, ngram_range=(2, 2))
toy_bigrams = toy_bigram_vectorizer.fit_transform(toy_docs)
print("\nLibrary bigram count matrix (CountVectorizer, ngram_range=(2, 2)):")
print(pd.DataFrame(toy_bigrams.toarray(), index=toy_names,
                   columns=toy_bigram_vectorizer.get_feature_names_out()).to_string())

library_counts = toy_bigrams[:, toy_bigram_vectorizer.vocabulary_[target_bigram]].toarray().ravel()
print(f"\nHand counts per document:    {list(hand_bigram_counts.values())}")
print(f"Library counts per document: {library_counts.tolist()}")
print("Match:", list(hand_bigram_counts.values()) == library_counts.tolist())

Step 1 - list the bigrams (pairs of neighbouring tokens) in each document:
  D1: ['movie not', 'not good']  ->  'not good' appears 1 time(s)
  D2: ['good movie', 'movie good', 'good act']  ->  'not good' appears 0 time(s)
  D3: ['bad movie', 'movie not', 'not good']  ->  'not good' appears 1 time(s)
  D4: ['great act']  ->  'not good' appears 0 time(s)

Step 2 - total count of 'not good' = 1 + 0 + 1 + 0 = 2

Library bigram count matrix (CountVectorizer, ngram_range=(2, 2)):
    bad movie  good act  good movie  great act  movie good  movie not  not good
D1          0         0           0          0           0          1         1
D2          0         1           1          0           1          0         0
D3          1         0           0          0           0          1         1
D4          0         0           0          1           0          0         0

Hand counts per document:    [1, 0, 1, 0]
Library counts per document: [1, 0, 1, 0]
Match: True
